# Walk-Forward Validation — 2025 Season

Simulates how model accuracy evolves as 2025 in-season data accumulates.

**Protocol:**
- Base training set: all 2021–2024 data
- Each step: add the next 2025 game date to training, evaluate on remaining 2025 dates
- X-axis: date in the 2025 season
- Y-axis: MAE on future 2025 games

In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src import config

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
HOLDOUT_SEASON     = 2025
N_ESTIMATORS       = 121       # Fixed as specified
MIN_TEST_ROWS      = 50        # Skip evaluation if fewer future rows remain
MAX_TRAIN_DATE     = '2025-08-31'  # Don't add data beyond this date (keeps test set meaningful)

XGB_PARAMS = {
    **config.XGB_PARAMS,
    'n_estimators': N_ESTIMATORS,
    'device': 'cpu',           # CPU avoids the libnvrtc dependency for notebooks
}

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv(config.PROCESSED_DATA_DIR / 'training_features.csv')
df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])

to_drop      = config.DROPPED_FEATURES + [config.TARGET_COL]
actual_drops = [c for c in to_drop if c in df.columns]

feature_cols = df.drop(columns=actual_drops).select_dtypes(include=['number']).columns.tolist()

base_df     = df[df['SEASON'] != HOLDOUT_SEASON].copy()
season_df   = df[df['SEASON'] == HOLDOUT_SEASON].copy().sort_values('GAME_DATE')
season_dates = sorted(season_df['GAME_DATE'].dt.normalize().unique())

print(f"Base training rows  : {len(base_df):,}  ({base_df['SEASON'].min()}–{base_df['SEASON'].max()})")
print(f"2025 game dates     : {len(season_dates)}")
print(f"2025 player rows    : {len(season_df):,}")

In [ ]:
# ── Walk-forward loop ─────────────────────────────────────────────────────────
results = []  # {date, n_train, n_test, mae_rest, mae_next}

max_date = pd.Timestamp(MAX_TRAIN_DATE)

# Baseline: model trained on pre-2025 only, evaluated on ALL of 2025
# Checkpoints stop at MAX_TRAIN_DATE so the test set stays meaningful
eligible_dates = [d for d in season_dates[:-1] if d <= max_date]
checkpoints = [None] + eligible_dates

for i, cutoff_date in enumerate(checkpoints):
    if cutoff_date is None:
        train_df  = base_df
        test_df   = season_df
        next_date = season_dates[0]
        label     = season_dates[0]   # plot at first date
    else:
        train_df  = pd.concat([base_df, season_df[season_df['GAME_DATE'].dt.normalize() <= cutoff_date]])
        test_df   = season_df[season_df['GAME_DATE'].dt.normalize() > cutoff_date]
        future_dates = [d for d in season_dates if d > cutoff_date]
        next_date = future_dates[0] if future_dates else None
        label     = cutoff_date

    if len(test_df) < MIN_TEST_ROWS:
        continue

    X_train = train_df[feature_cols].astype('float64').values
    y_train = train_df[config.TARGET_COL].values

    model = xgb.XGBRegressor(**XGB_PARAMS)
    model.fit(X_train, y_train, verbose=False)

    # MAE on rest of season
    X_test = test_df[feature_cols].astype('float64').values
    y_test = test_df[config.TARGET_COL].values
    mae_rest = mean_absolute_error(y_test, model.predict(X_test))

    # MAE on just the next game date
    if next_date is not None:
        next_df  = season_df[season_df['GAME_DATE'].dt.normalize() == next_date]
        X_next   = next_df[feature_cols].astype('float64').values
        y_next   = next_df[config.TARGET_COL].values
        mae_next = mean_absolute_error(y_next, model.predict(X_next)) if len(next_df) > 0 else None
    else:
        mae_next = None

    results.append({
        'date':     label,
        'n_train':  len(train_df),
        'n_test':   len(test_df),
        'mae_rest': mae_rest,
        'mae_next': mae_next,
    })
    print(f"  {str(label)[:10]}  |  train={len(train_df):5,}  test={len(test_df):4,}  rest={mae_rest:.3f}"
          + (f"  next={mae_next:.3f}" if mae_next else ""))

results_df = pd.DataFrame(results)
print(f"\n✅ Done — {len(results_df)} checkpoints evaluated.")

In [ ]:
# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(14, 5))

# Rest-of-season MAE
ax1.plot(results_df['date'], results_df['mae_rest'],
         color='steelblue', linewidth=2, label='MAE — rest of season')

# Next-date MAE (noisier, single slate)
next_valid = results_df.dropna(subset=['mae_next'])
ax1.plot(next_valid['date'], next_valid['mae_next'],
         color='darkorange', linewidth=1.2, linestyle=':', alpha=0.8, label='MAE — next date only')

ax1.axhline(6.71, color='tomato', linestyle='--', linewidth=1.2, label='Naive baseline (6.71)')
ax1.set_ylabel('MAE (Fantasy Points)')
ax1.set_xlabel('2025 Season Date')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax1.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Training size on secondary axis
ax2 = ax1.twinx()
ax2.fill_between(results_df['date'], results_df['n_train'],
                 alpha=0.10, color='seagreen', label='Train rows (right)')
ax2.set_ylabel('Training rows', color='seagreen')
ax2.tick_params(axis='y', labelcolor='seagreen')

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.title('Walk-Forward MAE — Rest of season vs. next date only', fontsize=13)
plt.tight_layout()
plt.savefig('walk_forward_2025.png', dpi=150)
plt.show()
print('Saved: walk_forward_2025.png')

In [ ]:
# ── Summary stats ─────────────────────────────────────────────────────────────
print(f"MAE at season start  (pre-2025 model only): {results_df['mae_rest'].iloc[0]:.3f}")
print(f"MAE at season end    (most 2025 data seen): {results_df['mae_rest'].iloc[-1]:.3f}")
print(f"Improvement over season                   : {results_df['mae_rest'].iloc[0] - results_df['mae_rest'].iloc[-1]:.3f} pts")
print(f"Naive baseline                            : 6.71")
print(f"Dates model beats baseline                : {(results_df['mae_rest'] < 6.71).sum()} / {len(results_df)}")